# MYZ307E — Improved CRNN Training
**Mesut Anlak** | 5-block ResNet + 2-layer BiLSTM + Temporal Attention

Run cells top-to-bottom. GPU runtime must be enabled (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Install dependencies (only what's not pre-installed in Colab) ─────
!pip install -q scikit-learn

In [ ]:
# ── 3. Copy model and training scripts from Drive to Colab runtime ───────
import shutil, os

DRIVE_PROJECT = '/content/drive/MyDrive/MYZ GRUP'

for fname in ['model_improved.py', 'train_improved.py']:
    src = os.path.join(DRIVE_PROJECT, fname)
    dst = os.path.join('/content', fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied {fname}')
    else:
        print(f'WARNING: {src} not found — upload the file to Drive first')

In [ ]:
# ── 4. Quick sanity check — forward pass ─────────────────────────────────
import sys
sys.path.insert(0, '/content')

import torch
from model_improved import build_model

model = build_model()
dummy = torch.zeros(4, 1, 128, 130)
with torch.no_grad():
    out = model(dummy)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Output shape      : {out.shape}')          # expect (4, 10)
print(f'Trainable params  : {total_params:,}')

In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────────────
# Adjust --data_dir to wherever Ahmet Selim's preprocessed_data folder lives.

DATA_DIR   = f'{DRIVE_PROJECT}/preprocessed_data'
SAVE_PATH  = f'{DRIVE_PROJECT}/best_improved_crnn.pth'

!python /content/train_improved.py \
    --data_dir   "{DATA_DIR}" \
    --save_path  "{SAVE_PATH}" \
    --epochs     30 \
    --lr         3e-4 \
    --batch_size 64 \
    --dropout    0.3 \
    --patience   5

In [ ]:
# ── 6. Plot training curves ───────────────────────────────────────────────
import json
import matplotlib.pyplot as plt
from pathlib import Path

history_path = str(Path(SAVE_PATH).with_name('improved_training_history.json'))
with open(history_path) as f:
    history = json.load(f)

epochs      = [h['epoch']      for h in history]
train_loss  = [h['train_loss'] for h in history]
val_loss    = [h['val_loss']   for h in history]
train_acc   = [h['train_acc']  for h in history]
val_acc     = [h['val_acc']    for h in history]

best_epoch = epochs[val_acc.index(max(val_acc))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, label='Train')
ax1.plot(epochs, val_loss,   label='Validation')
ax1.axvline(best_epoch, color='grey', linestyle='--', label=f'Best epoch ({best_epoch})')
ax1.set_title('Training and Validation Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-entropy Loss')
ax1.legend()

ax2.plot(epochs, train_acc, label='Train')
ax2.plot(epochs, val_acc,   label='Validation')
ax2.axvline(best_epoch, color='grey', linestyle='--', label=f'Best epoch ({best_epoch})')
ax2.set_title('Training and Validation Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plot_path = str(Path(SAVE_PATH).with_name('improved_training_curves.png'))
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved → {plot_path}')

In [ ]:
# ── 7. Plot confusion matrix ──────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, '/content')
from model_improved import build_model
from train_improved import GTZANDataset, load_data, GENRES, SEED

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = build_model().to(device)
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

file_paths, labels = load_data(DATA_DIR)
_, val_paths, _, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, stratify=labels, random_state=SEED)
val_ds     = GTZANDataset(val_paths, val_labels)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

all_preds, all_targets = [], []
with torch.no_grad():
    for x, y in val_loader:
        preds = model(x.to(device)).argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(y.tolist())

cm = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ticks = np.arange(len(GENRES))
ax.set_xticks(ticks); ax.set_yticks(ticks)
ax.set_xticklabels(GENRES, rotation=45, ha='right')
ax.set_yticklabels(GENRES)
for i in range(len(GENRES)):
    for j in range(len(GENRES)):
        ax.text(j, i, f'{cm_norm[i,j]:.2f}',
                ha='center', va='center',
                color='white' if cm_norm[i,j] > 0.5 else 'black', fontsize=7)
ax.set_xlabel('Predicted Genre'); ax.set_ylabel('True Genre')
ax.set_title('Normalized Confusion Matrix — Improved CRNN (Validation Set)')
plt.tight_layout()
cm_path = str(Path(SAVE_PATH).with_name('improved_confusion_matrix.png'))
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Saved → {cm_path}')